# Egyptian Real Estate Price Prediction: ML Pipeline
**Author:** Mohamed Tarek  
**Objective:** To develop a robust machine learning model for predicting property prices in Cairo and Giza using web-scraped data from Bayut. This project focuses on handling non-normal distributions and hyper-local feature engineering.

In [9]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score

## 1. Data Ingestion
Loading the raw dataset containing approximately 50,000 listings with features including location, property type, area, and bedroom/bathroom counts.

In [10]:
dataset = pd.read_csv('real_estate_data_bayut_full.csv')
dataset.head()

## 2. Initial Data Cleaning
Dropping high-cardinality links and redundant currency columns to focus the feature space on predictors.

In [11]:
dataset.drop(['link','currency'],axis = 1,inplace=True)
dataset.info()

## 3. Feature Engineering & Distribution Optimization
> **Technical Strategy:** 
> 1. **Location Splitting:** We decompose the `location` string into `region` and `city` to capture hyper-local pricing trends.
> 2. **Log Transformation:** Real estate prices follow a power-law distribution. We apply $y' = \ln(1 + y)$ to stabilize variance and minimize the influence of extreme luxury outliers.
> 3. **Outlier Mitigation:** Clipping data at the 1st and 99th percentiles to ensure the model generalizes across the standard market.

In [12]:
dataset.dropna(subset=['beds', 'baths'], inplace=True)
dataset.drop('plan_type',axis = 1,inplace = True)
dataset = dataset.reset_index()

dataset['down_payment'] = (
    dataset['down_payment']
    .fillna('0')                             
    .astype(str)                             
    .str.replace('EGP', '', regex=False)     
    .str.replace(',', '', regex=False)       
    .str.strip()                             
    .astype(float)                           
)

dataset['area'] = (
    dataset['area']
    .str.replace('Sq. M.','',regex = False)
    .str.replace(',', '', regex=False)       
    .str.strip()
    .astype(int)
)

dataset['price'] = (
    dataset['price']
    .str.replace(',', '', regex=False)       
    .str.strip()
    .astype(int)
)

dataset['dp_ratio'] = dataset['down_payment'] / dataset['price']
dataset['price'] = np.log1p(dataset['price'])
q_low = dataset["price"].quantile(0.01)
q_hi  = dataset["price"].quantile(0.99)
dataset = dataset[(dataset["price"] < q_hi) & (dataset["price"] > q_low)]
dataset = dataset.reset_index()

location = dataset['location'].tolist()
region = []
city = []

for i in location:
    parts = i.split(',')
    region.append(parts[0])
    city.append("".join(parts[1:]))

dataset['region'] = pd.Series(region)
dataset['city'] = pd.Series(city)
dataset.drop('location',axis = 1,inplace = True)

region_counts = dataset['region'].value_counts()
rare_regions = region_counts[region_counts < 10].index
dataset = dataset[~dataset['region'].isin(rare_regions)]

dataset['region'] = dataset['region'].astype('category')
dataset['city'] = dataset['city'].astype('category')
dataset['type'] = dataset['type'].astype('category')

## 4. Exploratory Data Analysis (EDA)
Visualizing the price distribution after log transformation to confirm near-normality and identify any remaining skewness.

In [13]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
sns.histplot(dataset['price'], bins=50, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Log-Transformed Price Distribution')
sns.boxplot(x=dataset['price'], ax=axes[1], color='salmon')
axes[1].set_title('Post-Clipping Outlier Detection')
plt.tight_layout()
plt.show()

## 5. Model Preparation
Utilizing a stratified split based on `region` to ensure training and validation sets are representative of the Egyptian geography.

In [ ]:
x = dataset.drop(['price','index'],axis = 1)
y= dataset['price']

x_train,x_temp,y_train,y_temp = train_test_split(x,y,test_size = 0.2, random_state=42,stratify = x['region']) 
x_valid,x_test,y_valid,y_test = train_test_split(x_temp,y_temp,test_size = 0.5, random_state=42,stratify = x_temp['region'])

## 6. Categorical Encoding
Converting property types into numerical features using One-Hot Encoding.

In [ ]:
enc = OneHotEncoder(sparse_output=False).set_output(transform = 'pandas')
cols = ['type']

encoded = enc.fit_transform(x_train[cols])
x_train = pd.concat([x_train.drop(cols,axis = 1),encoded],axis = 1)

encoded = enc.transform(x_valid[cols])
x_valid = pd.concat([x_valid.drop(cols,axis = 1),encoded],axis = 1)

encoded = enc.transform(x_test[cols])
x_test = pd.concat([x_test.drop(cols,axis = 1),encoded],axis = 1)

## 7. Model Architecture: XGBoost Regressor
Implementing **eXtreme Gradient Boosting** with hyperparameter constraints to prevent overfitting (e.g., `max_depth`, `subsample`).

In [16]:
model = XGBRegressor(
                        n_jobs = -1,
                        learning_rate = 0.05,
                        max_depth = 5,
                        subsample = 0.8,
                        colsample_bytree = 0.7,
                        colsample_bylevel = 0.8,
                        n_estimators=5000,
                        early_stopping_rounds=50,
                        reg_alpha = 2,
                        min_child_weight = 5,
                        tree_method="hist",      
                        enable_categorical=True, 
                        max_cat_to_onehot=5,                        
)

## 8. Training & Evaluation
Evaluating the model using MAE on back-transformed prices ($R^2$ scores shown below).

In [17]:
model.fit(x_train,y_train,eval_set=[(x_valid, y_valid)] )
prediction_train = model.predict(x_train)
prediction = model.predict(x_valid)
real_y = np.expm1(y_valid)
real_pred = np.expm1(prediction)
maye = mean_absolute_error(real_y, real_pred)
r2 = r2_score(y_valid,prediction)
r_train = r2_score(y_train,prediction_train)
mean = np.expm1(y_valid.mean())

print(f"MAE = {int(maye)}")
print(f"Validation r2 score = {int(r2*100)}%")
print(f"MAE to mean ratio = {int(maye / mean * 100)}%")

In [ ]:
prediction_final = model.predict(x_test)
real_y = np.expm1(y_test)
real_pred = np.expm1(prediction_final)
maye = mean_absolute_error(real_y, real_pred)
r2 = r2_score(y_test,prediction)
mean = np.expm1(y_train.mean())

print(f"MAE on test = {int(maye)}")
print(f"Test r2 score = {int(r2*100)}%")
print(f"MAE to mean ratio = {int(maye / mean * 100)}%")